# Baltimore Transit-Zone LVT — 4:1 Split-Rate Within One Mile of Rail Stations

Variant of the citywide Baltimore model (`cities/baltimore/model.ipynb`) restricted to parcels whose centroid lies within one mile of a rail station. Models a revenue-neutral 4:1 split-rate land value tax **within the transit zone only** — parcels outside the zone are out of scope and assumed unchanged.

Policy assumptions:

- Scope: **Baltimore city levy only**, same as the citywide model
- Universe: parcels with centroid within **1 mile** of any MDOT MTA rail station — Light RailLink (33 stations), Metro SubwayLink (14), and MARC commuter rail (44 statewide; only stations near the city contribute parcels)
- Reform: **4:1 split-rate**, revenue-neutral **within the zone** — zone parcels collectively pay the same total city tax as today, redistributed between land and improvements
- Existing exemptions preserved: fully exempt where there is no current city bill (`ARTAXBAS <= 0` or `CITY_TAX <= 0`); these parcels are **excluded** from the modeled universe, exports, and graphics
- Current tax = observed `CITY_TAX` (gross city levy before parcel-level credits), per Baltimore's derived-millage pattern
- Land/improvement split: `ARTAXBAS` allocated by current full cash shares (`CURRLAND`/`CURRIMPR`) — portal-verified basis (2026-06-11); the `BFCV*` base-cycle fields are NULL-land for ~46% of parcels and are not used

Data sources:

- Parcels: Baltimore City `Realproperty_OB` ArcGIS layer, reusing the citywide model's cache in `../baltimore/data/`
- Rail stations: MD iMAP hosted layers on ArcGIS Online (`services.arcgis.com/njFNhDsUCentVYJW`) — `Maryland_Transit_Light_Rail_Stations`, `Maryland_Transit_Metro_Subway_Stations`, `Maryland_Transit_MARC_Train_Stations`. The state server `geodata.md.gov` had an expired TLS certificate as of 2026-06-11, so the ArcGIS Online mirrors are used.
- Buffering is done in EPSG:26985 (NAD83 / Maryland, meters) for accurate distances; EPSG:3857 overstates distances ~27% at Baltimore's latitude.

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

CITY_NAME = 'baltimore_transit'
STATE_FIPS = '24'
COUNTY_FIPS = '510'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0
RAIL_BUFFER_MILES = 1.0
MD_STATE_PLANE = 'EPSG:26985'
METERS_PER_MILE = 1609.344

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Step 2: Load citywide parcel data

Reuses the citywide Baltimore caches (`baltimore_attrs_*.parquet`, `baltimore_geometry_*.parquet`) from `../baltimore/data/` when present; otherwise re-fetches from the `Realproperty_OB` FeatureServer into this city's `data/` directory.

In [ ]:
BALTIMORE_DATA = Path('../baltimore/data')
PARCEL_QUERY_URL = 'https://geodata.baltimorecity.gov/egis/rest/services/CityView/Realproperty_OB/FeatureServer/0/query'
ATTR_FIELDS = [
    'OBJECTID', 'PIN', 'CURRLAND', 'CURRIMPR', 'EXMPLAND', 'EXMPIMPR',
    'BFCVLAND', 'BFCVIMPR', 'ARTAXBAS', 'CITY_TAX', 'CITYCRED', 'CCREDAMT',
    'STATCRED', 'SCREDAMT', 'ZONECODE', 'USEGROUP', 'DHCDUSE1', 'DWELUNIT',
    'NO_IMPRV', 'VACIND', 'OWNER_1', 'OWNER_2', 'FULLADDR', 'NEIGHBOR',
    'PROPDESC', 'YEAR_BUILD', 'STRUCTAREA', 'LOT_SIZE',
]


def _latest(pattern):
    hits = sorted(list(DATA_DIR.glob(pattern)) + list(BALTIMORE_DATA.glob(pattern)), key=lambda p: p.name)
    return hits[-1] if hits else None


def fetch_arcgis_records(query_url, out_fields=None, chunk_size=1000, return_geometry=False):
    session = requests.Session()
    count = session.get(
        query_url, params={'f': 'json', 'where': '1=1', 'returnCountOnly': 'true'}, timeout=60
    ).json()['count']
    print(f'Total records: {count:,}')
    rows = []
    for offset in range(0, count, chunk_size):
        params = {
            'f': 'json', 'where': '1=1',
            'outFields': '*' if out_fields is None else ','.join(out_fields),
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': offset, 'resultRecordCount': chunk_size,
            'orderByFields': 'OBJECTID ASC',
        }
        if return_geometry:
            params.update({'outSR': 4326, 'geometryPrecision': 6})
        features = session.get(query_url, params=params, timeout=180).json().get('features', [])
        if not features:
            break
        rows.extend(features)
    return rows


attrs_cache = _latest('baltimore_attrs_*.parquet')
if attrs_cache is not None:
    parcel_attrs = pd.read_parquet(attrs_cache)
    print(f'Loaded attribute cache: {attrs_cache}')
else:
    feats = fetch_arcgis_records(PARCEL_QUERY_URL, out_fields=ATTR_FIELDS)
    parcel_attrs = (
        pd.DataFrame([f['attributes'] for f in feats])
        .drop_duplicates('OBJECTID').sort_values('OBJECTID').reset_index(drop=True)
    )
    parcel_attrs.to_parquet(DATA_DIR / 'baltimore_attrs_fresh.parquet', index=False)

geometry_cache = _latest('baltimore_geometry_*.parquet')
if geometry_cache is not None:
    parcel_geometry = gpd.read_parquet(geometry_cache)
    print(f'Loaded geometry cache: {geometry_cache}')
else:
    feats = fetch_arcgis_records(PARCEL_QUERY_URL, out_fields=['OBJECTID', 'PIN'], return_geometry=True)
    rows = []
    for f in feats:
        rings = f['geometry'].get('rings', [])
        geom = Polygon(rings[0], holes=rings[1:]) if rings else None
        rows.append({'OBJECTID': f['attributes']['OBJECTID'], 'PIN': f['attributes']['PIN'], 'geometry': geom})
    parcel_geometry = (
        gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
        .drop_duplicates('OBJECTID').sort_values('OBJECTID').reset_index(drop=True)
    )
    parcel_geometry.to_parquet(DATA_DIR / 'baltimore_geometry_fresh.parquet')

gdf = parcel_geometry.merge(parcel_attrs, on=['OBJECTID', 'PIN'], how='inner')
gdf = gdf.drop_duplicates(subset=['OBJECTID']).sort_values('OBJECTID').reset_index(drop=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
print(f'Citywide parcels: {len(gdf):,}')

In [ ]:
# Clean values and flag full exemptions (same logic as the citywide model)
numeric_cols = [
    'CURRLAND', 'CURRIMPR', 'EXMPLAND', 'EXMPIMPR', 'BFCVLAND', 'BFCVIMPR',
    'ARTAXBAS', 'CITY_TAX', 'CITYCRED', 'CCREDAMT', 'STATCRED', 'SCREDAMT',
    'DWELUNIT', 'YEAR_BUILD', 'STRUCTAREA',
]
for col in numeric_cols:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

text_cols = ['ZONECODE', 'USEGROUP', 'DHCDUSE1', 'NO_IMPRV', 'VACIND', 'OWNER_1', 'OWNER_2', 'FULLADDR', 'NEIGHBOR', 'PROPDESC']
for col in text_cols:
    if col in gdf.columns:
        gdf[col] = gdf[col].fillna('').astype(str).str.strip()

gdf['full_exmp'] = ((gdf['ARTAXBAS'] <= 0) | (gdf['CITY_TAX'] <= 0)).astype(int)

# Land/improvement basis: allocate the taxed base (ARTAXBAS) by current full cash shares.
# BFCVLAND is NULL for ~46% of parcels (verified against SDAT records 2026-06-11), so the
# BFCV base-cycle components cannot carry the land signal; CURRLAND/CURRIMPR match SDAT's
# current "Value" column exactly.
curr_total = gdf['CURRLAND'] + gdf['CURRIMPR']
land_share = (gdf['CURRLAND'] / curr_total).where(curr_total > 0)
gdf['taxable_land_value'] = (gdf['ARTAXBAS'].clip(lower=0) * land_share).fillna(0.0)
gdf['taxable_improvement_value'] = (gdf['ARTAXBAS'].clip(lower=0) * (1 - land_share)).fillna(0.0)
gdf['hold_out'] = (gdf['ARTAXBAS'] > 0) & (gdf['CITY_TAX'] > 0) & (curr_total <= 0)
print(f"Fully exempt (no current city bill): {int(gdf['full_exmp'].sum()):,} of {len(gdf):,}")
print(f"Billed parcels without a current land/improvement split (held at current tax): {int(gdf['hold_out'].sum()):,}")

# Per-account lot size parsed from the assessor's LOT_SIZE string ('14X83-10',
# '1.953 ACRES', '741 SF IMP. ONLY'). Parcel-polygon area is unreliable here:
# improvement-only accounts share the whole community's polygon (e.g. 1101 Horners Lane),
# so geometry-based areas overstate those lots by orders of magnitude.
import re

def parse_lot_size(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip().upper().replace(',', '')
    try:
        m = re.match(r'^(\d+(?:\.\d+)?)\s*ACRE', s)
        if m:
            return float(m.group(1)) * 43560.0
        m = re.match(r'^(\d+(?:\.\d+)?)\s*S\.?\s*Q?\.?\s*F', s)
        if m:
            return float(m.group(1))
        m = re.match(r'^(\d+)(?:-(\d+))?(?:\.\d+)?\s*X\s*(\d+)(?:-(\d+))?', s)
        if m:
            front = float(m.group(1)) + (float(m.group(2)) / 12.0 if m.group(2) else 0.0)
            depth = float(m.group(3)) + (float(m.group(4)) / 12.0 if m.group(4) else 0.0)
            return front * depth
    except (ValueError, TypeError):
        return np.nan
    return np.nan

gdf['lot_sqft'] = gdf['LOT_SIZE'].map(parse_lot_size)
print(f"Lot size parsed: {gdf['lot_sqft'].notna().mean()*100:.2f}% of parcels")

# SDAT 5-digit land use code — separate pull (not in the original cached field list).
# Codes 46000/46100/46200 are the parking family: commercial lots, surface parking,
# and detached residential garages on their own accounts (verified via parking-operator
# owner names, 2026-06-11). PIN maps 1:1 to a code.
sdat_cache = _latest("baltimore_sdatcode_*.parquet") if "_latest" in dir() else None
if sdat_cache is None:
    from pathlib import Path as _Path
    _hits = sorted(_Path("data").glob("baltimore_sdatcode_*.parquet")) + sorted(_Path("../baltimore/data").glob("baltimore_sdatcode_*.parquet"))
    sdat_cache = _hits[-1] if _hits else None
if sdat_cache is not None:
    sdat_codes = pd.read_parquet(sdat_cache)
    print(f"Loaded SDAT use-code cache: {sdat_cache}")
else:
    _feats = fetch_arcgis_records(PARCEL_QUERY_URL if "PARCEL_QUERY_URL" in dir() else parcel_query_url, out_fields=["OBJECTID", "PIN", "SDATCODE"])
    sdat_codes = pd.DataFrame([f["attributes"] for f in _feats]).drop_duplicates("OBJECTID")
    sdat_codes.to_parquet(DATA_DIR / "baltimore_sdatcode_fresh.parquet" if "DATA_DIR" in dir() else data_dir / "baltimore_sdatcode_fresh.parquet", index=False)
sdat_codes["SDATCODE"] = sdat_codes["SDATCODE"].fillna("").astype(str).str.strip()
gdf["SDATCODE"] = gdf["PIN"].map(sdat_codes.drop_duplicates("PIN").set_index("PIN")["SDATCODE"]).fillna("")
print(f"SDAT use-code coverage: {(gdf['SDATCODE'] != '').mean()*100:.1f}%")

## Step 2b: Rail stations and the one-mile zone

All three MDOT MTA rail modes are fetched statewide and buffered one mile in EPSG:26985. A parcel is in the zone when its centroid falls inside the buffer union — stations just outside the city line (e.g. Halethorpe MARC, light rail stations across the county border) still contribute city parcels this way.

In [ ]:
STATION_LAYERS = {
    'Light RailLink': 'Maryland_Transit_Light_Rail_Stations',
    'Metro SubwayLink': 'Maryland_Transit_Metro_Subway_Stations',
    'MARC': 'Maryland_Transit_MARC_Train_Stations',
}
STATIONS_PATH = DATA_DIR / 'md_rail_stations.gpq'

if STATIONS_PATH.exists():
    stations = gpd.read_parquet(STATIONS_PATH)
    print(f'Loaded {len(stations):,} stations from cache')
else:
    frames = []
    for mode, svc in STATION_LAYERS.items():
        url = (
            'https://services.arcgis.com/njFNhDsUCentVYJW/arcgis/rest/services/'
            f'{svc}/FeatureServer/0/query?where=1%3D1&outFields=Name&f=geojson'
        )
        layer = gpd.read_file(url)
        layer['transit_mode'] = mode
        frames.append(layer[['Name', 'transit_mode', 'geometry']])
        print(f'{mode}: {len(layer)} stations')
    stations = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry='geometry', crs='EPSG:4326')
    stations.to_parquet(STATIONS_PATH)

print(stations['transit_mode'].value_counts().to_string())

In [ ]:
# Build the one-mile buffer union and select parcels by centroid
stations_proj = stations.to_crs(MD_STATE_PLANE)
buffers = stations_proj.buffer(RAIL_BUFFER_MILES * METERS_PER_MILE)
zone_shape = buffers.union_all()

centroids = gdf.geometry.to_crs(MD_STATE_PLANE).centroid
centroid_gdf = gpd.GeoDataFrame({'parcel_idx': gdf.index}, geometry=centroids, crs=MD_STATE_PLANE)
zone_gdf = gpd.GeoDataFrame(geometry=[zone_shape], crs=MD_STATE_PLANE)
in_zone = gpd.sjoin(centroid_gdf, zone_gdf, how='inner', predicate='within')['parcel_idx']
gdf['in_transit_zone'] = gdf.index.isin(in_zone)

# Which stations actually contribute at least one Baltimore parcel
buffer_gdf = gpd.GeoDataFrame(stations_proj[['Name', 'transit_mode']].copy(), geometry=buffers, crs=MD_STATE_PLANE)
station_hits = gpd.sjoin(buffer_gdf, centroid_gdf, how='inner', predicate='contains')
contributing = station_hits.reset_index()[['Name', 'transit_mode']].drop_duplicates()
print(f'Stations with >= 1 Baltimore parcel within {RAIL_BUFFER_MILES:g} mile: {len(contributing)} of {len(stations)}')
print(contributing['transit_mode'].value_counts().to_string())

zone = gdf[gdf['in_transit_zone']].copy()
print(f'Zone parcels: {len(zone):,} of {len(gdf):,} citywide ({len(zone) / len(gdf) * 100:.1f}%)')
print(f"Zone share of citywide city levy: {zone['CITY_TAX'].sum() / gdf['CITY_TAX'].sum() * 100:.1f}%")

## Step 3: Property categories

Same Baltimore-specific hybrid mapping as the citywide model: `ZONECODE` supplies the zoning family, `USEGROUP` and `DWELUNIT` split residential by size, and parcels with no taxable improvements are `Vacant Land`.

In [ ]:
# Fully exempt parcels (no current city bill) are excluded from the modeled universe,
# exports, and report graphics. They pay $0 under both systems and only dilute the
# category and quintile statistics (e.g. ~61% of Vacant Land is city-owned exempt).
zone = zone[zone['full_exmp'] == 0].copy()
print(f"Modeling universe: {len(zone):,} billed zone parcels (fully exempt removed)")


def categorize_baltimore_property(row):
    zone_code = str(row.get('ZONECODE', '')).strip().upper()
    usegroup = str(row.get('USEGROUP', '')).strip().upper()
    dwell_units = pd.to_numeric(row.get('DWELUNIT', 0), errors='coerce')
    curr_impr = pd.to_numeric(row.get('CURRIMPR', 0), errors='coerce')
    lot_sqft = pd.to_numeric(row.get('lot_sqft'), errors='coerce')
    no_imprv = str(row.get('NO_IMPRV', '')).strip().upper()

    sdatcode = str(row.get('SDATCODE', '')).strip()
    if sdatcode in {'46000', '46100', '46200'}:
        return 'Transportation - Parking'

    if (curr_impr <= 0) or (no_imprv == 'Y'):
        return 'Vacant Land'

    if usegroup == 'R':
        if dwell_units <= 1:
            # 1/4 acre (10,890 sq ft) isolates the top ~4% of SFH lots (median lot is ~1,650 sq ft)
            if lot_sqft >= 10_890:
                return 'Single Family Residential (1/4+ acre)'
            return 'Single Family Residential'
        if 2 <= dwell_units <= 4:
            return 'Small Multi-Family (2-4 units)'
        if dwell_units >= 5:
            return 'Large Multi-Family (5+ units)'
        return 'Other Residential'

    if zone_code.startswith('R-'):
        return 'Other Residential'

    if zone_code.startswith('I-'):
        return 'Industrial'

    if zone_code.startswith('OR-') or zone_code.startswith('C-') or zone_code.startswith('EC-') or zone_code.startswith('CC') or zone_code.startswith('PC-') or zone_code.startswith('TOD-') or zone_code in {'BSC', 'IMU-1', 'MI', 'H', 'OS'}:
        return 'Commercial / Mixed Use'

    if usegroup in {'C', 'CC', 'CR', 'RC', 'EC'}:
        return 'Commercial / Mixed Use'

    if usegroup == 'I':
        return 'Industrial'

    if usegroup == 'U':
        return 'Utility / Special'

    if usegroup == 'E':
        return 'Institutional / Exempt'

    if usegroup == 'M':
        return 'Large Multi-Family (5+ units)'

    return 'Other'


zone['PROPERTY_CATEGORY'] = zone.apply(categorize_baltimore_property, axis=1)
display(
    zone['PROPERTY_CATEGORY']
    .value_counts(dropna=False)
    .rename_axis('PROPERTY_CATEGORY')
    .reset_index(name='parcel_count')
)

## Step 4: Current tax

The citywide derived millage (`CITY_TAX / ARTAXBAS`) is the validation anchor; observed `CITY_TAX` is used as `current_tax`, matching the citywide model. The revenue-neutral target is the zone's observed city levy.

In [ ]:
city_millage = round((gdf['CITY_TAX'].sum() / gdf['ARTAXBAS'].sum()) * 1000.0, 4)
print(f'Derived citywide millage: {city_millage:.4f} mills')

zone['city_millage'] = city_millage
target_revenue = float(zone['CITY_TAX'].sum())

modeled_revenue, _, zone = calculate_current_tax(
    df=zone,
    tax_value_col='ARTAXBAS',
    millage_rate_col='city_millage',
    exemption_flag_col='full_exmp',
)
zone['modeled_current_tax'] = zone['current_tax']
zone['current_tax'] = zone['CITY_TAX']

gap_pct = (modeled_revenue / target_revenue - 1) * 100
print(f'Modeled from ARTAXBAS x millage: ${modeled_revenue:,.0f}')
print(f'Observed CITY_TAX (zone target):  ${target_revenue:,.0f}')
print(f'Gap: {gap_pct:+.2f}%')
assert abs(gap_pct) < 5.0, f'Revenue gap {gap_pct:.1f}% exceeds threshold'

## Step 5: 4:1 split-rate within the zone

Revenue-neutral solve over zone parcels only, on the taxable components (`BFCVLAND`, `BFCVIMPR`). Fully exempt parcels are excluded from the solve and stay at zero.

In [ ]:
land_millage, improvement_millage, new_revenue, zone = model_split_rate_tax(
    df=zone,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=target_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_flag_col='full_exmp',
    exclude_mask=zone['hold_out'],
)

print(f'Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}')
print(f'Revenue check: ${new_revenue:,.0f} vs target ${target_revenue:,.0f}')

category_summary = calculate_category_tax_summary(
    df=zone,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    pct_threshold=10.0,
)
print_category_tax_summary(
    summary_df=category_summary,
    title='4:1 Split-Rate Within One Mile of Rail Stations - Baltimore, MD',
    pct_threshold=10.0,
)
display(category_summary)

## Step 6: Exploration charts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
med = zone.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
ax.barh(med.index, med.values, color=np.where(med.values > 0, '#8B0000', '#228B22'))
ax.axvline(0, color='black', linewidth=1, linestyle='dotted')
ax.set_xlabel('Median % change')
ax.set_title('Baltimore transit zone - median tax change % by category (4:1 split-rate)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(9, 10))
zone_proj = zone.to_crs(MD_STATE_PLANE)
pts = zone_proj.geometry.centroid
ax.scatter(pts.x, pts.y, c=np.where(zone_proj['tax_change'] > 0, '#8B0000', '#228B22'), s=0.5, alpha=0.5)
gpd.GeoSeries([zone_shape], crs=MD_STATE_PLANE).boundary.plot(ax=ax, color='black', linewidth=0.7)
stations_proj.plot(ax=ax, color='#1f4e9c', markersize=14, marker='^')
xmin, ymin, xmax, ymax = zone_proj.total_bounds
pad = 1500
ax.set_xlim(xmin - pad, xmax + pad)
ax.set_ylim(ymin - pad, ymax + pad)
ax.set_axis_off()
ax.set_title('Transit zone: parcels by tax-change direction (red = increase)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'zone_map.png', dpi=150)
plt.close()
print('Saved exploration charts to data/')

## Step 7: Census join and standard export

In [ ]:
# From here on, `gdf` is the modeled transit-zone dataframe (standard closing pattern)
gdf = zone

# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — spatial join adds them above.
            # Do NOT do a second gdf.merge(census_data) here: census_gdf has the columns
            # baked in, so a second merge creates median_income_x/y duplicates and silently
            # zeros out all demographic output.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

## Validation Summary (re-based 2026-06-11)

| Check | Result |
|---|---|
| Revenue match | −0.83% (ARTAXBAS × derived 22.2914 mills vs observed zone `CITY_TAX` $569.6M) |
| Zone selection | 119,037 of 238,492 city parcels (49.9%); 33 of 91 stations contribute parcels |
| Modeled universe | 105,898 billed parcels (fully exempt excluded from models, exports, and graphics) |
| Revenue neutrality | Exact: modeled $569,570,529 = target |
| Land / improvement millage | 53.0422 / 13.2605 (zone taxable land share 23.2%) |
| Census coverage | 100.0% matched |
| Report PNGs | 7 of 7 |
| SFH (<1/4 acre) median change | +1.4% |
| SFH (1/4+ acre) median change | +3.1% (n=1,767) |

Basis and interpretation:

- **Land/improvement basis (portal-verified 2026-06-11):** the taxed base `ARTAXBAS` is allocated by current full cash shares (`CURRLAND`/`CURRIMPR`), which match SDAT's current "Value" column exactly on every parcel checked. The earlier run used the `BFCV*` base-cycle fields, whose land component is NULL for ~46% of parcels — that artifact produced the previously reported SFR median of +35.2%; the corrected figure is +1.4%.
- With the corrected basis, a within-zone 4:1 split-rate is roughly neutral for single-family parcels. The burden moves onto taxable vacant land (+125% of its current levy) and low-improvement industrial parcels (median +18%), while improvement-heavy commercial sheds in dollar terms (−6.6% of category total, −$15.0M).
- Billed parcels without a current land/improvement split (TIF land units and similar; 606 citywide) are held at their current tax via `exclude_mask` rather than redistributed.
- Parcels are flagged fully exempt when they carry no current city bill (`ARTAXBAS <= 0` or `CITY_TAX <= 0`).